# Phase 4: Regression ModelsPredicting the exact attendance percentage using Scikit-Learn pipelines.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score


## 1. Load DataWe must drop target leakage columns (Students_Present).

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')

leaky_cols = ['Students_Present', 'Attendance_Class', 'Total_Enrolled', 'Date']
X_train = train_df.drop(columns=[c for c in leaky_cols + ['Attendance_Percentage'] if c in train_df.columns])
y_train = train_df['Attendance_Percentage']
X_val = val_df.drop(columns=[c for c in leaky_cols + ['Attendance_Percentage'] if c in val_df.columns])
y_val = val_df['Attendance_Percentage']


## 2. Preprocessing Pipeline

In [ ]:
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])


## 3. Train Best Model (XGBoost)

In [ ]:
from sklearn.pipeline import Pipeline
model = XGBRegressor(random_state=42)
pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', model)])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_val)
print(f'MAE: {mean_absolute_error(y_val, y_pred):.2f}')
print(f'R2: {r2_score(y_val, y_pred):.2f}')
